# Prototipo: simulación fundamentada + línea base
### Proyecto Integrador — Machine Learning I — Primera entrega
**Sebastián Ramos · Daniel Cipagauta**

Este notebook respalda la Sección 3 (Estrategia de datos) y la Sección 4.3 (Línea base) del informe. Genera un conjunto de datos simulado de localización interior por RSSI WiFi (siguiendo el modelo de propagación log-normal shadowing citado en el artículo QA-kNN), construye las características por cuartiles (Q1, Q2, Q3) propuestas en el artículo, y confirma el comportamiento esperado de una línea base de clasificador de mayoría. El ajuste riguroso de hiperparámetros y la comparación formal de modelos se hará en la Segunda entrega; aquí solo se valida que el pipeline es viable de principio a fin.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

## 1. Escenario simulado: piso de un edificio universitario colombiano

Piso de 30 m x 20 m, con 18 zonas dispuestas en una cuadrícula de 3 x 6 y 8 puntos de acceso (APs) distribuidos por corredores, siguiendo la Sección 3.2 del informe.

In [ ]:
# --- Geometria del piso ---
BUILDING_W, BUILDING_D = 30.0, 20.0  # metros
N_ROWS, N_COLS = 3, 6
N_ZONES = N_ROWS * N_COLS  # 18 zonas

# Centro de cada zona (cuadricula regular)
zone_positions = {}
zid = 0
for r in range(N_ROWS):
    for c in range(N_COLS):
        x = (c + 0.5) * (BUILDING_W / N_COLS)
        y = (r + 0.5) * (BUILDING_D / N_ROWS)
        zone_positions[f"Zona_{zid+1:02d}"] = (x, y)
        zid += 1

# --- Posicion de los 8 APs: distribuidos por corredores/esquinas ---
ap_positions = {
    "AP1": (2.0, 2.0),   "AP2": (15.0, 2.0),  "AP3": (28.0, 2.0),
    "AP4": (2.0, 18.0),  "AP5": (15.0, 18.0), "AP6": (28.0, 18.0),
    "AP7": (15.0, 10.0), "AP8": (8.0, 10.0),
}
N_APS = len(ap_positions)

print(f"Zonas: {N_ZONES}, APs: {N_APS}")

## 2. Simulación fundamentada: modelo log-normal shadowing

RSSI(d) = RSSI(d0) − 10·n·log10(d/d0) + Xσ, con Xσ ~ N(0, σ²), tal como se describe en la Sección 3.1 del informe (y en la Sección 3.2 del artículo QA-kNN). El exponente de pérdida de trayecto (n) sube levemente en zonas con más obstáculos (oficinas cerradas) respecto a zonas abiertas (pasillos), y se asume cobertura completa de los 8 APs desde cualquier zona.

In [ ]:
RSSI_D0 = -45.0   # dBm a 1 metro (referencia tipica WiFi 2.4GHz)
D0 = 1.0          # metros
SIGMA_SHADOWING = 6.0  # dB, ruido de sombra (4-8 dB tipico en interiores)

def path_loss_exponent(zone_id):
    """Zonas de borde/esquina (oficinas cerradas) -> mas obstaculos -> mayor n.
    Zonas centrales (pasillos) -> menos obstaculos -> menor n."""
    idx = int(zone_id.split("_")[1])
    col = (idx - 1) % N_COLS
    es_pasillo_central = col in (2, 3)  # columnas centrales simulan corredor
    return 2.6 if es_pasillo_central else 3.3

def simulate_rssi_reading(zone_id, ap_id, rng):
    zx, zy = zone_positions[zone_id]
    ax, ay = ap_positions[ap_id]
    d = max(np.hypot(zx - ax, zy - ay), D0)
    n_pl = path_loss_exponent(zone_id)
    rssi_det = RSSI_D0 - 10 * n_pl * np.log10(d / D0)
    noise = rng.normal(0, SIGMA_SHADOWING)
    return rssi_det + noise

# Sanity check: rango de RSSI esperado (debe parecerse a un router real: -30 a -90 dBm)
muestras = [simulate_rssi_reading("Zona_01", "AP1", rng) for _ in range(500)]
print(f"RSSI simulada (Zona_01, AP1): media={np.mean(muestras):.1f} dBm, "
      f"min={np.min(muestras):.1f}, max={np.max(muestras):.1f}")

## 3. Extracción de características por cuartiles (Q1, Q2, Q3)

Siguiendo la Sección 3.3 del artículo: cada instancia agrupa `M_LECTURAS` lecturas RSSI por AP y las resume en sus tres cuartiles, formando un vector de 3×N_APS características.

In [ ]:
M_LECTURAS = 40          # lecturas RSSI por AP en cada instancia (visita), como en el articulo
VISITAS_POR_ZONA = 280   # ~18 zonas x 280 = ~5040 registros (>= 5000 requeridos)

def construir_instancia(zone_id, rng):
    """Genera M_LECTURAS lecturas por AP y devuelve los cuartiles como features."""
    fila = {}
    for ap_id in ap_positions:
        lecturas = np.array([simulate_rssi_reading(zone_id, ap_id, rng) for _ in range(M_LECTURAS)])
        q1, q2, q3 = np.percentile(lecturas, [25, 50, 75])
        fila[f"{ap_id}_Q1"] = q1
        fila[f"{ap_id}_Q2"] = q2
        fila[f"{ap_id}_Q3"] = q3
    fila["zona"] = zone_id
    return fila

registros = []
for zone_id in zone_positions:
    for _ in range(VISITAS_POR_ZONA):
        registros.append(construir_instancia(zone_id, rng))

df = pd.DataFrame(registros)
print("Dimensiones del dataset:", df.shape)
df.head()

In [ ]:
# Verificacion: >= 5000 registros y clases balanceadas (requisito de la guia del proyecto)
assert df.shape[0] >= 5000, "Se requieren al menos 5000 registros"
print("Total de registros:", df.shape[0])
print("\nDistribucion de clases (deben estar balanceadas):")
print(df["zona"].value_counts())

## 4. Línea base: clasificador de mayoría

Con 18 zonas balanceadas, la exactitud esperada de esta línea base es ≈1/18 ≈ 5.6%. Se confirma aquí, sobre esta muestra simulada preliminar, como referencia mínima que cualquier modelo entrenado deberá superar.

In [ ]:
X = df.drop(columns=["zona"])
y = df["zona"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)

linea_base = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
linea_base.fit(X_train, y_train)
y_pred_base = linea_base.predict(X_test)

acc_base = accuracy_score(y_test, y_pred_base)
print(f"Exactitud de la linea base (clasificador de mayoria): {acc_base:.1%}")
print(f"Exactitud esperada teorica (1/{N_ZONES} zonas): {1/N_ZONES:.1%}")

## 5. Chequeo preliminar de viabilidad (kNN, sin ajuste riguroso)

Solo para confirmar que el pipeline funciona de principio a fin y que el problema es aprendible con las técnicas del curso. El ajuste formal de `k`, la validación cruzada y la comparación con los demás modelos candidatos (Naive Bayes, regresión logística) se harán en la Segunda entrega.

In [ ]:
pipeline_knn = Pipeline([
    ("escalado", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])
pipeline_knn.fit(X_train, y_train)
y_pred_knn = pipeline_knn.predict(X_test)

print(f"Exactitud kNN (k=5, sin ajustar): {accuracy_score(y_test, y_pred_knn):.1%}")
print(f"F1-macro kNN: {f1_score(y_test, y_pred_knn, average='macro'):.3f}")
print(f"Mejora sobre la linea base: {accuracy_score(y_test, y_pred_knn) - acc_base:+.1%}")

**Conclusión del prototipo:** el pipeline simulación → cuartiles → clasificación corre de principio a fin, el dataset cumple el mínimo de 5000 registros balanceados, y la línea base se comporta como lo predice la teoría (~5.6%). El chequeo preliminar de kNN ya muestra una mejora sustancial sobre la línea base, lo que confirma que vale la pena continuar con el plan de modelamiento de la Sección 4 en la Segunda entrega.